# cvprofiles — IRT as a SCORE-upstream scoring technology

The engine is **score-agnostic**: any scalar column per unit can be a candidate measure,
however it was built. Item response theory (IRT) is one principled scoring technology that
lives *upstream* of the engine — it turns binary item responses into scalar person scores
$\hat\theta_i$, one number per unit, which then become cvprofiles measure columns.

This notebook shows the full loop with a **hand-rolled 1PL (Rasch) fit** — no IRT package,
just numpy + scipy, so the scoring step is auditable:
1. simulate a latent trait and binary item responses;
2. fit a 1PL model to recover person scores;
3. feed those scores (plus a naive sum-score and a noisy measure) into a cvprofiles profile;
4. see which operationalizations survive the researcher-authored network.

Scientific stance: the engine never scores — *you* decide how to fill score columns. IRT is
one defensible choice; the nomological network then disciplines it like any other measure.



In [ ]:
from __future__ import annotations
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.optimize as opt
import yaml

import cvprofiles
from cvprofiles.pipeline import run_profile

print("cvprofiles", cvprofiles.__version__)



## 1. Simulate a latent trait and 1PL item responses

A 1PL (Rasch) model: person $i$ has latent score $\theta_i$; item $j$ has difficulty $b_j$;
the probability of a correct/endorsed response is

$$P(x_{ij}=1 \mid \theta_i, b_j) = \mathrm{logit}(\theta_i - b_j).$$

We generate $n=300$ persons, $J=30$ items, and a latent trait that also drives an external
auxiliary (so the network has something to anchor on).



In [ ]:
rng = np.random.default_rng(20260806)
n, J = 300, 30

# latent trait and an external auxiliary that co-moves with it
theta = rng.normal(size=n)
v_aux = 0.8 * theta + 0.6 * rng.normal(size=n)

# item difficulties spread across the trait range
b = np.linspace(-1.5, 1.5, J)

# 1PL responses
logit = theta[:, None] - b[None, :]
p = 1.0 / (1.0 + np.exp(-logit))
X = (rng.random((n, J)) < p).astype(float)

print("response matrix:", X.shape, "mean endorsement:", round(float(X.mean()), 3))



## 2. Hand-rolled 1PL fit (joint ML)

We maximize the joint log-likelihood over $\theta_i$ and $b_j$ (recentering $\theta$ for the
standard identifiability convention). Initialization matters: item difficulties start from
empirical endorsement rates, person scores from the sum-score logit. The fitted
$\hat\theta_i$ is our **IRT measure column**. A naive **sum score** (fraction endorsed) is
the classic model-free alternative, and a noisy "LLM-ish" score gives the menu a weak member.



In [ ]:
def fit_1pl(X, max_iter=1000, tol=1e-7):
    n, J = X.shape
    # empirical-difficulty init: b0 = -logit(endorsement); theta from sum-score logit
    end = np.clip(X.mean(axis=0), 1e-3, 1 - 1e-3)
    b0 = -np.log(end / (1 - end))
    z = np.log((X.mean(axis=1) + 1e-3) / (1 - X.mean(axis=1) + 1e-3))
    z = (z - z.mean()) / z.std() * 1.5

    def unpack(params):
        return params[:n], params[n:]

    def neg_ll(params):
        th, bj = unpack(params)
        m = th[:, None] - bj[None, :]
        lse = np.maximum(m, 0) + np.log1p(np.exp(-np.abs(m)))
        return float(np.sum(lse) - np.sum(X * m))

    x0 = np.concatenate([z, b0])
    res = opt.minimize(neg_ll, x0, method="L-BFGS-B",
                       options={"maxiter": max_iter, "gtol": tol})
    th, bj = unpack(res.x)
    th = th - th.mean()
    bj = bj + th.mean()  # keep logit(theta - b) invariant
    return th, bj, res

theta_irt, b_fit, res = fit_1pl(X)
sum_score = X.mean(axis=1)                                   # naive operationalization
noisy = 0.3 * theta + 1.0 * rng.normal(size=n)               # weak, noisy measure

print("fit converged:", res.success)
print("corr(theta_true, theta_irt):", round(float(np.corrcoef(theta, theta_irt)[0, 1]), 3))
print("corr(theta_true, sum_score):", round(float(np.corrcoef(theta, sum_score)[0, 1]), 3))



## 3. Build the cvprofiles inputs

The fitted scores are just columns. The researcher-authored network says: *a valid measure of
this trait must correlate at least $\theta=0.35$ with the external auxiliary, in the stated
positive direction.* The target is the correlation of each measure with the outcome $y$.



In [ ]:
work = Path(tempfile.mkdtemp(prefix="cvp_irt_"))

y = 0.5 * theta + rng.normal(size=n)  # outcome depends on the trait

scores = pd.DataFrame({
    "unit_id": [f"u{i:04d}" for i in range(n)],
    "m_irt": theta_irt,
    "m_sum": sum_score,
    "m_noisy": noisy,
    "v_aux": v_aux,
    "y": y,
})
scores.to_csv(work / "scores.csv", index=False)

roles = {
    "unit_id": "unit_id",
    "measures": ["m_irt", "m_sum", "m_noisy"],
    "aux": ["v_aux"],
    "outcome": "y",
    "diagnostic": [],
}
(work / "roles.json").write_text(json.dumps(roles))

network = {
    "schema_version": "1",
    "name": "irt_oracle",
    "delta": 0.0,
    "restrictions": [
        {"id": "r_corr_min_aux", "type": "corr_min", "theta": 0.35,
         "params": {"variable": "v_aux"}},
        {"id": "r_corr_sign_aux", "type": "corr_sign", "theta": 0.0,
         "params": {"variable": "v_aux", "sign": 1}},
    ],
}
(work / "network.yaml").write_text(yaml.safe_dump(network, sort_keys=False))

beta = {"schema_version": "1", "type": "corr_y", "outcome": "y", "params": {}}
(work / "beta.yaml").write_text(yaml.safe_dump(beta, sort_keys=False))

print("inputs written to", work)



In [ ]:
result = run_profile(
    scores=work / "scores.csv",
    roles=work / "roles.json",
    network=work / "network.yaml",
    beta=work / "beta.yaml",
    out_dir=work / "run",
    seed=0,
    title="IRT-as-scoring profile",
)

print("run_id :", result.run_id)
print("M*     :", result.identify.admissible)
print("[L,U]  :", result.identify.range_L, result.identify.range_U)
print("rejected:", result.identify.rejected)



## 4. Read the profile

The **IRT score and the sum score both survive** the network: they co-move with the external
auxiliary strongly enough. The **noisy measure is rejected** — it fails the minimum
association bar. The headline range $[L,U]$ is the image of the target functional on the
survivors only; the noisy measure never enters it.

Notice what the network *did not* do: it did not judge IRT versus sum-score on intrinsic
grounds. It disciplined both through the same external implications. The honest comparison in
this DGP is that the two scoring technologies recover the latent trait similarly; IRT adds
item-level difficulty calibration and a latent metric, while the sum score is model-free and
trivially auditable. The menu, the network, and the thresholds are yours — the engine treats
any scalar column the same way.



In [ ]:
# self-checking assertions
assert set(result.identify.admissible) == {"m_irt", "m_sum"}
assert "m_noisy" in result.identify.rejected
assert result.identify.range_L is not None and result.identify.range_L <= result.identify.range_U
assert result.identify.empty is False
print("all IRT tutorial assertions passed")



## What to look at next

- The run directory holds the full audit trail: `report.html`, `report.json`, slacks,
  admissible set, range — the same artifacts as any other profile.
- IRT is one upstream scorer. Dictionary scores, LLM scores, PCA factors all slot in the same
  way: scalar columns in, disciplined by the network.
- The engine remains score-agnostic and model-free. Scoring, the menu, and the nomological
  network are researcher-owned.

